## **Installs**

In [ ]:
#!pip install -U transformers

## **Hugging Face Token**

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("my_hf_token")

In [2]:
from huggingface_hub import login
login(token=hf_token)

## **Imports**

In [3]:
import json
import re
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc, traceback

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [4]:
MODEL_NAME = "google/gemma-2b"
HF_TOKEN = hf_token

## **Load Model**

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    dtype=torch.float16,      # NOT torch_dtype (deprecated, as you already found)
    device_map="auto",
)
model.eval()

DEVICE = next(model.parameters()).device
NUM_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded {MODEL_NAME}: {NUM_LAYERS} layers, hidden_dim={HIDDEN_DIM}, device={DEVICE}")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Loaded google/gemma-2b: 18 layers, hidden_dim=2048, device=cuda:0


In [6]:
#freeze model params

for p in model.parameters():
    p.requires_grad_(False)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params in base model (should be 0): {trainable}")

Trainable params in base model (should be 0): 0


## **Dataset Loading**

In [7]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset.csv")  # update path if running locally
# If running outside Kaggle, point this at your local copy instead, e.g.:
# df = pd.read_csv("pico_dataset.csv")
print(df.shape)
df.head(3)

(50, 7)


,srno,clinical_speciality,research_question,P,I,C,O
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...


## **Generate Negative Responses**

In [8]:
GEN_MAX_NEW_TOKENS = 100
GEN_DO_SAMPLE = False
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95

@torch.no_grad()
def generate_negative(question, max_new_tokens=GEN_MAX_NEW_TOKENS):
    prompt = f"{question.strip()}\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=GEN_DO_SAMPLE,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return text.strip()

In [9]:
negative_responses = []

for i, row in df.iterrows():
    neg = generate_negative(row["research_question"])
    negative_responses.append(neg)
    if (i + 1) % 5 == 0 or (i + 1) == len(df):
        print(f"[{i+1}/{len(df)}] done")
        print("  Q:", row["research_question"][:100])
        print("  neg:", neg[:150].replace("\n", " "))

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[5/50] done
  Q: [Diagnostic] What is the diagnostic test accuracy of coronary computed tomography angiography (CCTA)
  neg: [Diagnostic] What is the diagnostic test accuracy of coronary computed tomography angiography (CCTA) compared with invasive coronary angiography (frac
[10/50] done
  Q: [Prognostic] Is high tumor mutational burden (TMB ≥ 10 mutations/megabase), compared with low TMB ($
  neg: [Prognostic] Is high tumor mutational burden (TMB ≥ 10 mutations/megabase), compared with low TMB ($<$10 mut/Mb), associated with improved overall sur
[15/50] done
  Q: [Diagnostic] What is the diagnostic test accuracy of cerebrospinal fluid (CSF) Aβ42​/Aβ40​ ratio com
  neg: [Diagnostic] What is the diagnostic accuracy of CSF Aβ42​/Aβ40​ ratio compared with amyloid PET imaging for diagnosing Alzheimer's disease in patients
[20/50] done
  Q: [Prognostic] Is an elevated serum procalcitonin level ($ \ge 0.5ng/mL)atpresentation,comparedwithlow
  neg: [Prognostic] Is an elevated serum procalcito

In [10]:
df_out = df.copy()
df_out["negative_response"] = negative_responses

out_path = "/kaggle/working/pico_dataset_with_negatives.csv"
df_out.to_csv(out_path, index=False)
print(f"Saved {df_out.shape} to {out_path}")
df_out.head(3)

Saved (50, 8) to /kaggle/working/pico_dataset_with_negatives.csv


,srno,clinical_speciality,research_question,P,I,C,O,negative_response
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...,"The study was a randomized, double-blind, plac..."
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...,"This is a randomized, double-blind, placebo-co..."
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...,The aim of this study was to evaluate the impa...


## **Part 2: Training Steering Vector**

## **Loading Positive and Negative Dataset**

In [11]:
df = pd.read_csv("/kaggle/input/datasets/anarvaaa/dataset-for-ssv/pico_dataset_with_negatives.csv")  # update path if running locally
# If running outside Kaggle, point this at your local copy instead, e.g.:
# df = pd.read_csv("pico_dataset_with_negatives.csv")
print(df.shape)
df.head(3)

(50, 8)


,srno,clinical_speciality,research_question,P,I,C,O,negative_response
0,1,Cardiology,In adults with heart failure with preserved ej...,Adults with HFpEF (LVEF ≥ 50%),"SGLT2 inhibitors (e.g., empagliflozin, dapagli...",Placebo + standard care,Composite of cardiovascular death and hospital...,"The study was a randomized, double-blind, plac..."
1,2,Cardiology,In patients undergoing percutaneous coronary i...,Adults undergoing PCI for acute coronary syndrome,Short-term DAPT (1–3 mo) followed by P2Y12 inh...,Standard DAPT for 12 months,Bleeding Academic Research Consortium (BARC) t...,"This is a randomized, double-blind, placebo-co..."
2,3,Cardiology,In adults with severe asymptomatic aortic sten...,Adults with severe asymptomatic aortic stenosis,Early elective AVR (surgical or transcatheter),Conservative watchful waiting,All-cause mortality; myocardial infarction at ...,The aim of this study was to evaluate the impa...


## **Train-Test Split and Prompt, Positive and Negative Triplet**

In [12]:
def format_positive(row):
    return (
        f"P: {row['P']}\n"
        f"I: {row['I']}\n"
        f"C: {row['C']}\n"
        f"O: {row['O']}"
    )

def format_prompt(row):
    return f"{row['research_question'].strip()}\n"

examples = []
for _, row in df.iterrows():
    examples.append({
        "prompt": format_prompt(row),
        "positive": format_positive(row) + tokenizer.eos_token,
        "negative": str(row["negative_response"]).strip() + tokenizer.eos_token,
    })

random.shuffle(examples)
n_val = max(5, int(0.1 * len(examples)))
val_examples = examples[:n_val]
train_examples = examples[n_val:]
print(f"train={len(train_examples)}  val={len(val_examples)}")
print("--- sample ---")
print("PROMPT:", train_examples[0]["prompt"])
print("POSITIVE:", train_examples[0]["positive"])
print("NEGATIVE:", train_examples[0]["negative"])

train=45  val=5
--- sample ---
PROMPT: In adults aged 45 and older with degenerative medial meniscus tears and no severe osteoarthritis, does arthroscopic partial meniscectomy compared with structured physical therapy improve patient-reported pain and function at 12 months?

POSITIVE: P: Adults aged ≥45 with symptomatic degenerative medial meniscus tears
I: Arthroscopic partial meniscectomy (APM)
C: Structured physical therapy/conservative exercise programs
O: Patient-reported knee pain and physical function (e.g., KOOS or WOMAC scores)<eos>
NEGATIVE: The purpose of this study was to compare the effectiveness of a 12-week structured physical therapy program with a 12-week arthroscopic partial meniscectomy program in patients with degenerative medial meniscus tears and no severe osteoarthritis.
The study was a randomized, controlled trial. Patients were randomly assigned to either the physical therapy group or the arthroscopic partial meniscectomy group. The primary outcome was pain mea

## **Steering Vector**

In [13]:
class SteeringVector(nn.Module):
    """Rank-1 steering vector: h <- h + alpha * w1 + b1  (eq. 9 in RePS paper)."""
    def __init__(self, dim, dtype=torch.float32):
        super().__init__()
        self.w1 = nn.Parameter(torch.randn(dim, dtype=dtype) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(dim, dtype=dtype))

    def forward(self, h, alpha):
        # keep params in fp32 (see exp1 NaN fix) and cast to h's device/dtype
        # only at the point of injection into the model's (fp16) activations.
        w1 = self.w1.to(device=h.device, dtype=h.dtype)
        b1 = self.b1.to(device=h.device, dtype=h.dtype)
        return h + alpha * w1 + b1


class SteeringHook:
    """Forward hook that injects a SteeringVector into a decoder layer's output."""
    def __init__(self, steering_vector: SteeringVector):
        self.sv = steering_vector
        self.enabled = False
        self.alpha = 0.0
        self.handle = None

    def __call__(self, module, inputs, output):
        if not self.enabled:
            return output
        if isinstance(output, tuple):
            hidden_states = self.sv(output[0], self.alpha)
            return (hidden_states,) + output[1:]
        else:
            return self.sv(output, self.alpha)

    def attach(self, layer_module):
        self.handle = layer_module.register_forward_hook(self)
        return self.handle

    def remove(self):
        if self.handle is not None:
            self.handle.remove()
            self.handle = None

In [14]:
STEER_LAYER = max(1, int(NUM_LAYERS * 0.4))
layer_device = next(model.model.layers[STEER_LAYER].parameters()).device
print(f"Hooking layer index {STEER_LAYER} / {NUM_LAYERS} (on {layer_device})")

steering_vector = SteeringVector(HIDDEN_DIM, dtype=torch.float32).to(layer_device)
steering_hook = SteeringHook(steering_vector)
steering_hook.attach(model.model.layers[STEER_LAYER])

n_trainable = sum(p.numel() for p in steering_vector.parameters() if p.requires_grad)
print(f"Trainable steering-vector params: {n_trainable}  (2 x {HIDDEN_DIM})")

Hooking layer index 7 / 18 (on cuda:1)
Trainable steering-vector params: 4096  (2 x 2048)


## **Preference Objective**

In [15]:
def seq_logprob(prompt_ids, response_ids, alpha, device):
    """Length-normalized log p(response | prompt) under the steered model."""
    input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0).to(device)
    attention_mask = torch.ones_like(input_ids)

    steering_hook.enabled = True
    steering_hook.alpha = alpha
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = out.logits  # (1, seq_len, vocab)

    shift_logits = logits[:, :-1, :].float()
    shift_labels = input_ids[:, 1:]
    log_probs = F.log_softmax(shift_logits, dim=-1)
    token_logp = log_probs.gather(2, shift_labels.unsqueeze(-1)).squeeze(-1)  # (1, seq_len-1)

    P = prompt_ids.shape[0]
    R = response_ids.shape[0]
    resp_logp = token_logp[:, P - 1 : P - 1 + R]  # only the response tokens
    return resp_logp.sum() / R  # mean log-prob per token (SimPO-style length norm)


def build_pair(example, device):
    prompt_ids = tokenizer(example["prompt"], return_tensors="pt", add_special_tokens=True).input_ids[0]
    pos_ids = tokenizer(example["positive"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    neg_ids = tokenizer(example["negative"], return_tensors="pt", add_special_tokens=False).input_ids[0]
    return prompt_ids, pos_ids, neg_ids

## **Training**

In [16]:
TRAIN_FACTORS = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0]
LR = 0.02
EPOCHS = 15           # upper bound -- early stopping below will usually stop well before this
GRAD_CLIP = 1.0
BETA = 1.0             # sharpness of the preference loss
WEIGHT_DECAY = 0.05    # pulls the vector back toward 0 every step -- prevents runaway drift
MAX_VEC_NORM = 3.0     # hard cap on ||w1|| applied after every optimizer step
PATIENCE = 2           # stop if val_loss doesn't improve for this many epochs

# NOTE on the previous run: val_margin blew up to +10..+14 nats/token within a
# single epoch and kept climbing every epoch after, even once val_loss was
# already ~0. That's not "learning PICO structure" -- it's the optimizer
# exploiting an unconstrained, reference-free objective on a tiny (45-example)
# dataset to find a degenerate shortcut direction, and then continuing to
# drift along it because AdamW's adaptive step size doesn't shrink just
# because the true gradient did. weight_decay + a hard norm cap + early
# stopping directly address that.

optimizer = torch.optim.AdamW(steering_vector.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def preference_loss(example, alpha, device):
    prompt_ids, pos_ids, neg_ids = build_pair(example, device)
    logp_pos = seq_logprob(prompt_ids, pos_ids, alpha, device)
    logp_neg = seq_logprob(prompt_ids, neg_ids, alpha, device)
    margin = logp_pos - logp_neg
    loss = -F.logsigmoid(BETA * margin)
    return loss, margin.item()

@torch.no_grad()
def evaluate(examples, alpha, device):
    margins, losses = [], []
    for ex in examples:
        prompt_ids, pos_ids, neg_ids = build_pair(ex, device)
        logp_pos = seq_logprob(prompt_ids, pos_ids, alpha, device)
        logp_neg = seq_logprob(prompt_ids, neg_ids, alpha, device)
        margin = (logp_pos - logp_neg).item()
        margins.append(margin)
        losses.append(-F.logsigmoid(torch.tensor(BETA * margin)).item())
    acc = float(np.mean([m > 0 for m in margins]))  # how often positive is preferred
    return float(np.mean(losses)), float(np.mean(margins)), acc

@torch.no_grad()
def clip_vector_norm(sv, max_norm=MAX_VEC_NORM):
    norm = sv.w1.norm()
    if norm > max_norm:
        sv.w1.mul_(max_norm / norm)

In [17]:
history = {"train_loss": [], "val_loss": [], "val_margin": [], "val_acc": [], "vec_norm": []}
skipped_steps = 0
best_val_loss = float("inf")
best_state = None
epochs_since_improve = 0

for epoch in range(EPOCHS):
    random.shuffle(train_examples)
    epoch_losses = []
    for ex in train_examples:
        alpha = random.choice(TRAIN_FACTORS)

        optimizer.zero_grad()
        loss, _ = preference_loss(ex, alpha, layer_device)

        if not torch.isfinite(loss):
            skipped_steps += 1
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(steering_vector.parameters(), GRAD_CLIP)
        optimizer.step()
        clip_vector_norm(steering_vector)  # hard cap ||w1|| -- prevents runaway drift
        epoch_losses.append(loss.item())

    train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    val_loss, val_margin, val_acc = evaluate(val_examples, alpha=TRAIN_FACTORS[len(TRAIN_FACTORS)//2], device=layer_device)
    vec_norm = steering_vector.w1.norm().item()
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_margin"].append(val_margin)
    history["val_acc"].append(val_acc)
    history["vec_norm"].append(vec_norm)
    print(f"epoch {epoch+1:02d}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}  ||w1||={vec_norm:.3f}  (skipped {skipped_steps})")

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_state = copy.deepcopy(steering_vector.state_dict())
        epochs_since_improve = 0
    else:
        epochs_since_improve += 1
        if epochs_since_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no val_loss improvement for {PATIENCE} epochs).")
            break

# restore the best checkpoint rather than whatever the last (possibly-drifted) epoch produced
if best_state is not None:
    steering_vector.load_state_dict(best_state)
    print(f"Restored best checkpoint (val_loss={best_val_loss:.4f}, ||w1||={steering_vector.w1.norm().item():.3f})")

epoch 01/12  train_loss=0.1643  val_loss=0.0004  val_margin=+10.0723  val_pref_acc=1.00  (skipped 0)
epoch 02/12  train_loss=0.0052  val_loss=0.0021  val_margin=+7.1717  val_pref_acc=1.00  (skipped 0)
epoch 03/12  train_loss=0.0014  val_loss=0.0010  val_margin=+10.4504  val_pref_acc=1.00  (skipped 0)
epoch 04/12  train_loss=0.0003  val_loss=0.0007  val_margin=+11.6524  val_pref_acc=1.00  (skipped 0)
epoch 05/12  train_loss=0.0001  val_loss=0.0005  val_margin=+12.2793  val_pref_acc=1.00  (skipped 0)
epoch 06/12  train_loss=0.0002  val_loss=0.0004  val_margin=+13.0192  val_pref_acc=1.00  (skipped 0)
epoch 07/12  train_loss=0.0001  val_loss=0.0003  val_margin=+13.4095  val_pref_acc=1.00  (skipped 0)
epoch 08/12  train_loss=0.0001  val_loss=0.0003  val_margin=+13.4209  val_pref_acc=1.00  (skipped 0)
epoch 09/12  train_loss=0.0001  val_loss=0.0002  val_margin=+13.8770  val_pref_acc=1.00  (skipped 0)
epoch 10/12  train_loss=0.0001  val_loss=0.0002  val_margin=+14.4720  val_pref_acc=1.00  (sk

## **Best Steering Vector**

In [18]:
# Smaller, finer-grained range than before. IMPORTANT: we do NOT pick the
# factor with the highest margin anymore -- margin keeps increasing with alpha
# almost by construction (bigger intervention -> more separation), so "max
# margin" just selects the most extreme, least fluent factor every time. That
# was the bug that picked alpha=4.0 and produced token-salad generations.
# Instead we pick the *smallest* alpha that already achieves strong preference
# accuracy -- the minimal nudge that reliably works, which is much more likely
# to preserve fluency.
candidate_factors = [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]
factor_stats = {}
for f in candidate_factors:
    val_loss, val_margin, val_acc = evaluate(val_examples, alpha=f, device=layer_device)
    factor_stats[f] = {"loss": val_loss, "margin": val_margin, "acc": val_acc}
    print(f"alpha={f:>5.2f}  val_loss={val_loss:.4f}  val_margin={val_margin:+.4f}  val_pref_acc={val_acc:.2f}")

ACC_THRESHOLD = 0.8
qualifying = [f for f in candidate_factors if factor_stats[f]["acc"] >= ACC_THRESHOLD]
if qualifying:
    BEST_ALPHA = min(qualifying)
else:
    BEST_ALPHA = max(candidate_factors, key=lambda f: factor_stats[f]["acc"])
print(f"\nBest steering factor: {BEST_ALPHA}  "
      f"(smallest factor with val_pref_acc >= {ACC_THRESHOLD}, or best available)")
print("Sanity-check this with actual generations below -- log-prob margin alone")
print("does not guarantee fluent output; always eyeball a few samples.")

alpha=  1.0  val_loss=0.1846  val_margin=+7.4074  val_pref_acc=1.00
alpha=  2.0  val_loss=0.0000  val_margin=+14.1463  val_pref_acc=1.00
alpha=  4.0  val_loss=0.0000  val_margin=+14.5920  val_pref_acc=1.00
alpha=  6.0  val_loss=0.0000  val_margin=+14.4172  val_pref_acc=1.00
alpha=  8.0  val_loss=0.0001  val_margin=+14.1208  val_pref_acc=1.00
alpha= 10.0  val_loss=0.0002  val_margin=+13.4558  val_pref_acc=1.00
alpha= 12.0  val_loss=0.0005  val_margin=+12.4475  val_pref_acc=1.00
alpha= 16.0  val_loss=0.0016  val_margin=+10.3630  val_pref_acc=1.00
alpha= 20.0  val_loss=0.0041  val_margin=+8.6536  val_pref_acc=1.00

Best steering factor: 4.0


## **Generation**

In [19]:
@torch.no_grad()
def generate(prompt, alpha=None, max_new_tokens=100):
    steering_hook.enabled = alpha is not None
    steering_hook.alpha = alpha if alpha is not None else 0.0

    inputs = tokenizer(prompt, return_tensors="pt").to(layer_device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    steering_hook.enabled = False
    return text

In [20]:
for ex in val_examples[:3]:
    print("=" * 80)
    print("PROMPT:", ex["prompt"].strip())
    print("-" * 80)
    print("UNSTEERED:")
    print(generate(ex["prompt"], alpha=None))
    print("-" * 80)
    print(f"STEERED (learned vector, alpha={BEST_ALPHA}):")
    print(generate(ex["prompt"], alpha=BEST_ALPHA))
    print("-" * 80)
    print("POSITIVE (ground truth):")
    print(ex["positive"].replace(tokenizer.eos_token, ""))

PROMPT: In adults with out-of-hospital cardiac arrest (OHCA) and an initial shockable rhythm, does real-time audiovisual feedback during CPR compared with standard CPR without feedback improve survival to hospital discharge?
--------------------------------------------------------------------------------
UNSTEERED:
The study was a randomized controlled trial conducted in 10 hospitals in the United States. The study included 1,000 patients with OHCA and an initial shockable rhythm. Patients were randomly assigned to receive either real-time audiovisual feedback during CPR or standard CPR without feedback. The primary outcome was survival to hospital discharge.
The study found that patients who received real-time audiovisual feedback during CPR had a 10% higher survival to hospital discharge compared with patients who received standard CPR
--------------------------------------------------------------------------------
STEERED (learned vector, alpha=4.0):
</sup>:
Flashcards
Equ
Flashcard

In [ ]:
torch.save(
    {
        "w1": steering_vector.w1.detach().cpu(),
        "b1": steering_vector.b1.detach().cpu(),
        "layer": STEER_LAYER,
        "best_alpha": BEST_ALPHA,
        "model_name": MODEL_NAME,
        "objective": "preference (positive vs negative)",
    },
    "/kaggle/working/pico_steering_vector_exp2.pt",
)
print("Saved /kaggle/working/pico_steering_vector_exp2.pt")